# DAD Protocol: Dynamic Affinity Docking
## User-input Colab workflow for protein-ligand docking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyunho-song09/dad-protocol/blob/main/DAD_protocol.ipynb)

**Purpose:** paste your own protein sequence/FASTA and ligand SMILES, then generate docking-ready outputs and ranked results.  
**Default path:** custom input -> cached/uploaded PDB or automatic AF2/ESMFold structure -> pocket box -> ligand 3D -> GNINA docking -> result tables/figures.  
**Reuse:** keep the same `job_name` and change only `custom_ligand_smiles` to rescore new ligands against the cached PDB.

### Quick Start

| Step | Action |
|---|---|
| 1 | Open in Colab. GPU is recommended for AF2 and GNINA. |
| 2 | Run `0. Setup Environment`. If the kernel restarts, run all cells again. |
| 3 | Paste your protein sequence or FASTA in `custom_protein_fasta`. |
| 4 | Paste one or more SMILES entries in `custom_ligand_smiles`. |
| 5 | Keep `STRUCTURE_MODE="auto"`, or use `existing_or_upload` with your own PDB. |
| 6 | Download `docking_results.tsv` or the result zip. |

### Input Formats

Protein sequence without FASTA header is accepted:

```text
MRNMSIFMKVMVIVLILALGMIVIGVYSTFAL...
```

FASTA is also accepted:

```text
>ProteinA
MRNMSIFMKVMVIVLILALGMIVIGVYSTFAL...
```

SMILES can be unnamed, named, or batched:

```text
CC[C@H](C)[C@@H](C(=O)O)NC(=O)[C@H](C)N
Ala-Ile:CC[C@H](C)[C@@H](C(=O)O)NC(=O)[C@H](C)N
LigandA:CCO;LigandB:CCN
```

### Outputs

| Stage | Output |
|---|---|
| Input | normalized FASTA and SMILES files |
| Triage | accepted protein regions |
| Structure | cached, uploaded, ESMFold, or ColabFold AF2 PDB |
| Pocket | auto box center and box size |
| Ligand | 3D SDF files |
| Docking | GNINA SDF/log and score table |
| Results | ranked TSV, summary tables, figures, reproducibility JSON |


In [ ]:
#@title 0. Setup Environment {display-mode: "form"}
#@markdown **Time:** 5-10 min on first Colab run. ColabFold/GNINA can take longer.
#@markdown **Output:** progress bar, package check, optional GNINA check.
#@markdown If condacolab installs, Colab restarts the kernel. Run all cells again after restart.

INSTALL_GNINA = True  #@param {type:"boolean"}
INSTALL_COLABFOLD = False  #@param {type:"boolean"}

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec('google.colab') is not None
print(f'Runtime: {"Google Colab" if IN_COLAB else "Local/WSL2"}')

class DADProgress:
    def __init__(self, label: str, total: int):
        self.label = label
        self.total = max(1, int(total))
        self.current = 0
        self.render('start')

    def render(self, message: str = ''):
        width = 28
        filled = int(width * self.current / self.total)
        bar = '#' * filled + '-' * (width - filled)
        pct = int(100 * self.current / self.total)
        print(f'\r{self.label}: [{bar}] {pct:3d}% {self.current}/{self.total} {message}', end='')
        if self.current >= self.total:
            print()

    def update(self, message: str = ''):
        self.current = min(self.total, self.current + 1)
        self.render(message)

    def done(self, message: str = 'done'):
        self.current = self.total
        self.render(message)

def dad_progress(label: str, total: int) -> DADProgress:
    return DADProgress(label, total)

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def run_cmd(cmd, label: str, required: bool = True):
    print(f'\n{label}...')
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        tail = (result.stderr or result.stdout or '').strip().splitlines()[-25:]
        print('\n'.join(tail))
        if required:
            raise RuntimeError(f'{label} failed with exit code {result.returncode}')
    return result

p = dad_progress('Setup Environment', 8)
RESTART_REQUESTED = False

if IN_COLAB and shutil.which('mamba') is None:
    p.update('install condacolab')
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], 'Install condacolab')
    import condacolab
    print('\nInstalling Conda runtime. Colab will restart the kernel.')
    condacolab.install()
    RESTART_REQUESTED = True
    p.done('restart requested')
    print('After Colab reconnects, click Runtime > Run all. This is expected.')

if not RESTART_REQUESTED:
    p.update('conda runtime ready')

    pip_packages = ['pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn', 'py3Dmol']
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q'] + pip_packages, 'Install core Python packages')
    p.update('core packages')

    needs_bio = (not has_module('rdkit')) or (not has_module('Bio'))
    if needs_bio:
        if IN_COLAB and shutil.which('mamba'):
            try:
                run_cmd(['mamba', 'install', '-q', '-y', '-c', 'conda-forge', 'rdkit', 'biopython'],
                        'Install RDKit/Biopython with mamba')
            except Exception:
                print('mamba failed; trying pip fallback for RDKit/Biopython.')
                run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'biopython'],
                        'Install RDKit/Biopython with pip')
        else:
            run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit', 'biopython'],
                    'Install RDKit/Biopython with pip')
    p.update('chemistry packages')

    if INSTALL_COLABFOLD:
        print('\nColabFold will install on demand in the Structure step using the AlphaFold2 notebook backend.')
    else:
        print('\nStructure backend: auto mode uses cached/uploaded PDB, ESMFold API, then ColabFold AF2 fallback.')
    p.update('structure backend')

    if IN_COLAB and INSTALL_GNINA:
        cuda_packages = [
            'nvidia-cuda-runtime-cu12==12.9.79',
            'nvidia-cublas-cu12==12.9.2.10',
            'nvidia-cusparse-cu12==12.5.10.65',
            'nvidia-cufft-cu12==11.4.1.4',
            'nvidia-cusolver-cu12==11.7.5.82',
            'nvidia-cudnn-cu12==9.21.1.3',
            'nvidia-nvtx-cu12==12.9.79',
            'nvidia-cuda-nvrtc-cu12==12.9.86',
            'nvidia-nccl-cu12',
        ]
        run_cmd([sys.executable, '-m', 'pip', 'install', '-q'] + cuda_packages, 'Install CUDA runtime wheels')
        import site
        site_pkgs = site.getsitepackages()[0]
        lib_paths = [
            f'{site_pkgs}/nvidia/cudnn/lib',
            f'{site_pkgs}/nvidia/cuda_runtime/lib',
            f'{site_pkgs}/nvidia/cublas/lib',
            f'{site_pkgs}/nvidia/cusparse/lib',
            f'{site_pkgs}/nvidia/cufft/lib',
            f'{site_pkgs}/nvidia/cusolver/lib',
        ]
        existing = [path for path in lib_paths if Path(path).exists()]
        old_ld = os.environ.get('LD_LIBRARY_PATH', '')
        os.environ['LD_LIBRARY_PATH'] = ':'.join(existing + ([old_ld] if old_ld else []))
    p.update('CUDA libraries')

    GNINA_BIN = Path('/usr/local/bin/gnina')
    GNINA_URL = 'https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2'
    GNINA_VERSION_EXPECTED = 'gnina v1.3.2'

    if INSTALL_GNINA:
        if not GNINA_BIN.exists():
            run_cmd(['wget', '-q', '-O', str(GNINA_BIN), GNINA_URL], 'Download GNINA v1.3.2')
            GNINA_BIN.chmod(0o755)
        result = subprocess.run([str(GNINA_BIN), '--version'], capture_output=True, text=True)
        version_out = (result.stdout + result.stderr).strip()
        print(f'\nGNINA: {version_out[:100]}')
        if result.returncode != 0 or GNINA_VERSION_EXPECTED not in version_out:
            raise RuntimeError('GNINA version check failed. Use a Linux/Colab GPU runtime or rerun Setup.')
        print('GNINA check: PASS')
    else:
        print('\nGNINA install skipped.')
    p.update('GNINA')

    import rdkit
    import Bio
    p.update('import check')
    p.done('complete')
    print('Package check: PASS')


In [ ]:
#@title 1. Input Data {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** normalized protein and ligand files.

exec_mode = "live"  #@param ["live", "prepare_only"]
job_name = "DAD_run"  #@param {type:"string"}

custom_protein_fasta = "MRNMSIFMKVMVIVLILALGMIVIGVYSTFALRNNITEKTMQNLKALAENSGENLVSFIEQHTKLIDMLSRDANVMGVYKNEHEEDVWMKKLFNTVLKSYPDVMYVYVGLKDKRMYLIPETELPEGYDPTIRPWYQAAVAKPGQVIITEPYADASTGQLVVTVAKTIQTDEGIVGVVALDFDISKLSEKLMTKGKELGYLNAVVSKEGNIIMHSDKTLVGKNVANEEFFKKWMSGDESGVFGYTLNGVKRISGYKRLSNGWIFATLVLEKSLMKDVNRATFINISITIIAILLGIVVALFVTRGFVVKPISFLVEAAERIANGDLTTKIDYTAKDEIGKLATALSKMVNSLREIALNIERDSATVKQEASQVAAVSEEVSATIEELTAQVDNVNTNVNNASAAIEEMTSGIEEVAASAQNVAHASQKLSEEAQKVSNLANEGQKAILSISDVIVQTRQKADATFRIVEQLSESAKNIGEIVDTINSIAEQTNLLALNAAIEAARAGEAGRGFAVVADEIRKLAEESKQATQNIANILRGIVDSSMKASEATKETVEIVNKAYSESDLVKSQFEQILQSIVRMSQMTENLAASAQEQSAAAEEMSSAMDSASKSMVSVVEQMNEVTMAIKQQADAISNVARTVENLDNIAEKLVETVRRFKI"  #@param {type:"string"}
custom_ligand_smiles = "CC[C@H](C)[C@@H](C(=O)O)NC(=O)[C@H](C)N"  #@param {type:"string"}

MOUNT_DRIVE = False  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/DAD_runs"  #@param {type:"string"}

import os
import re
from pathlib import Path

if 'dad_progress' not in globals():
    class DADProgress:
        def __init__(self, label, total): self.label, self.total, self.current = label, total, 0
        def update(self, message=''):
            self.current += 1; print(f'{self.label}: {self.current}/{self.total} {message}')
        def done(self, message='done'): print(f'{self.label}: {message}')
    def dad_progress(label, total): return DADProgress(label, total)

p = dad_progress('Input Data', 5)

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

EXEC_MODE = exec_mode

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path(DRIVE_OUTPUT_DIR)
else:
    BASE_DIR = Path('/content') if IN_COLAB else Path.cwd()

safe_job_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', job_name).strip('_') or 'DAD_run'
WORK_DIR = BASE_DIR / safe_job_name
for sub in ['proteins', 'ligands', 'ligands_3d', 'structures', 'pockets', 'docking', 'results', 'results/tables', 'results/figures']:
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)
p.update('workspace')

AA_ALLOWED = set('ACDEFGHIKLMNPQRSTVWYBXZJUO')

def sanitize_id(text: str, fallback: str) -> str:
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(text)).strip('_')
    return text or fallback

def parse_custom_proteins(raw: str) -> dict:
    raw = (raw or '').strip()
    if not raw:
        return {}
    chunks = [chunk.strip() for chunk in raw.split(';') if chunk.strip()]
    proteins = {}
    for idx, chunk in enumerate(chunks, start=1):
        chunk = chunk.replace('\\n', '\n').strip()
        if chunk.startswith('>'):
            lines = [line.strip() for line in chunk.splitlines() if line.strip()]
            header = lines[0][1:].strip() or f'Protein_{idx}'
            seq = ''.join(line for line in lines[1:] if not line.startswith('>'))
        else:
            header = f'Protein_{idx}' if len(chunks) > 1 else 'Protein_1'
            seq = chunk
        seq = re.sub(r'[^A-Za-z]', '', seq).upper()
        invalid = sorted(set(seq) - AA_ALLOWED)
        if invalid:
            raise ValueError(f'Invalid amino-acid letters in {header}: {invalid}')
        if len(seq) < 20:
            raise ValueError(f'Protein sequence too short for {header}: {len(seq)} aa')
        prot_id = sanitize_id(header.split()[0], f'Protein_{idx}')
        proteins[prot_id] = f'>{prot_id}\n{seq}'
    return proteins

def parse_ligands(raw: str) -> dict:
    raw = (raw or '').strip()
    if not raw:
        return {}
    items = []
    for block in raw.split(';'):
        for line in block.splitlines():
            line = line.strip()
            if line:
                items.append(line)
    ligands = {}
    for idx, item in enumerate(items, start=1):
        if ':' in item and not item.startswith(('http://', 'https://')):
            left, right = item.split(':', 1)
            if re.fullmatch(r'[A-Za-z0-9_. -]+', left.strip()) and right.strip():
                name, smiles = left, right
            else:
                name, smiles = f'Ligand_{idx}', item
        elif ',' in item:
            left, right = item.split(',', 1)
            if any(ch in left for ch in '=#[]()/\\@+-'):
                smiles, name = left, right
            else:
                name, smiles = left, right
        else:
            parts = item.split()
            if len(parts) >= 2 and not any(ch in parts[0] for ch in '=#[]()/\\@+-'):
                name, smiles = parts[0], parts[1]
            else:
                name, smiles = f'Ligand_{idx}', item
        name = sanitize_id(name, f'Ligand_{idx}')
        smiles = smiles.strip()
        if not smiles:
            raise ValueError(f'Empty SMILES for {name}')
        ligands[name] = smiles
    return ligands

PROTEINS = parse_custom_proteins(custom_protein_fasta)
p.update('proteins parsed')
LIGANDS = parse_ligands(custom_ligand_smiles)
p.update('ligands parsed')

if not PROTEINS:
    raise ValueError('No proteins parsed. Paste a raw sequence or FASTA.')
if not LIGANDS:
    raise ValueError('No ligands parsed. Paste SMILES or name:SMILES.')

for prot_id, fasta in PROTEINS.items():
    (WORK_DIR / 'proteins' / f'{prot_id}.fasta').write_text(fasta + '\n')
(WORK_DIR / 'ligands' / 'ligands.smi').write_text('\n'.join(f'{smi}\t{name}' for name, smi in LIGANDS.items()) + '\n')
p.update('files written')

print(f'Proteins: {list(PROTEINS.keys())}')
print(f'Ligands : {list(LIGANDS.keys())}')
print(f'Cases   : {len(PROTEINS) * len(LIGANDS)}')
print(f'Mode    : {EXEC_MODE}')
print(f'Workdir : {WORK_DIR}')
p.done('complete')


In [ ]:
#@title 2. Triage: Sequence QC {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** accepted protein regions for docking.

import re
from dataclasses import dataclass
from typing import List

p = dad_progress('Triage', max(1, len(PROTEINS) + 1))

@dataclass
class TriageResult:
    orf_id: str
    raw_seq: str
    status: str
    verdict: str
    n_tm: int = 0
    has_signal_peptide: bool = False
    dock_region_start: int = 1
    dock_region_end: int = 0
    functional_class: str = 'user_input'
    notes: str = ''

    @property
    def dock_seq(self):
        seq = ''.join(line for line in self.raw_seq.splitlines() if not line.startswith('>'))
        end = self.dock_region_end if self.dock_region_end > 0 else len(seq)
        return seq[self.dock_region_start - 1:end]

def triage_sequence(orf_id: str, fasta_str: str) -> TriageResult:
    lines = fasta_str.strip().splitlines()
    seq = ''.join(line for line in lines if not line.startswith('>')).replace(' ', '').upper()
    length = len(seq)
    if length < 50:
        return TriageResult(orf_id, fasta_str, 'EXCLUDE', 'exclude', notes=f'too_short:{length}')
    has_sp = False
    sp_cut = 0
    if seq.startswith('M') and length > 30:
        hydrophobic = sum(1 for aa in seq[1:25] if aa in 'AVILMFYW')
        if hydrophobic >= 10:
            has_sp = True
            sp_cut = 22
    status = 'PASS_CLIPPED' if has_sp else 'PASS'
    notes = 'signal_peptide_clipped' if has_sp else 'accepted'
    return TriageResult(
        orf_id=orf_id,
        raw_seq=fasta_str,
        status=status,
        verdict='accept',
        has_signal_peptide=has_sp,
        dock_region_start=sp_cut + 1,
        dock_region_end=length,
        notes=notes,
    )

triage_records: List[TriageResult] = []
for orf_id, fasta in PROTEINS.items():
    rec = triage_sequence(orf_id, fasta)
    triage_records.append(rec)
    p.update(orf_id)

accepted = [record for record in triage_records if record.verdict == 'accept']
if not accepted:
    raise RuntimeError('No accepted proteins after triage.')

for record in triage_records:
    print(f'{record.orf_id}: {record.status}, dock_region={record.dock_region_start}-{record.dock_region_end}, {record.notes}')
p.done(f'{len(accepted)}/{len(triage_records)} accepted')


In [ ]:
#@title 3. Structure Input or Prediction {display-mode: "form"}
#@markdown **Time:** cached/uploaded PDB <1 min; ESMFold minutes/protein; ColabFold AF2 can take 10-30+ min/protein.
#@markdown **Output:** PDB files in `structures/` and a reusable structure cache.

STRUCTURE_MODE = "auto"  #@param ["auto", "existing_or_upload", "esmfold_api", "colabfold"]
EXISTING_PDB_DIR = ""  #@param {type:"string"}
REUSE_EXISTING_STRUCTURES = True  #@param {type:"boolean"}
STRUCTURE_CACHE_DIR = ""  #@param {type:"string"}
ESMFOLD_API_MAX_AA = 550  #@param {type:"integer"}
ALLOW_COLABFOLD_FALLBACK = True  #@param {type:"boolean"}
AF2_PRESET = "balanced"  #@param ["fast", "balanced", "high_accuracy"]
AF2_MODEL_TYPE = "auto"  #@param ["auto", "alphafold2_ptm", "alphafold2", "alphafold2_multimer_v3"]
AF2_NUM_RECYCLES = "auto"  #@param ["auto", "0", "1", "3", "6", "12", "24", "48"]
AF2_NUM_MODELS = 5  #@param [1, 2, 3, 4, 5] {type:"raw"}
AF2_NUM_SEEDS = 1  #@param [1, 2, 4, 8] {type:"raw"}
AF2_RELAX_TOP_N = 0  #@param [0, 1] {type:"raw"}

import glob
import hashlib
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

p = dad_progress('Structure', max(1, len(accepted) * 3 + 2))
STRUCT_DIR = WORK_DIR / 'structures'
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

if STRUCTURE_CACHE_DIR.strip():
    CACHE_DIR = Path(STRUCTURE_CACHE_DIR).expanduser()
elif MOUNT_DRIVE and IN_COLAB:
    CACHE_DIR = Path(DRIVE_OUTPUT_DIR) / '_structure_cache'
else:
    CACHE_DIR = WORK_DIR / 'structure_cache'
if REUSE_EXISTING_STRUCTURES:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

structure_paths = {}
structure_manifest = []

def structure_sequence(record) -> str:
    seq = getattr(record, 'dock_seq', '') or ''.join(
        line.strip() for line in record.raw_seq.splitlines() if not line.startswith('>')
    )
    return ''.join(ch for ch in seq.upper() if ch.isalpha())

def sequence_hash(seq: str) -> str:
    return hashlib.sha256(seq.encode('utf-8')).hexdigest()[:12]

def pdb_has_atoms(path: Path) -> bool:
    try:
        text = path.read_text(errors='ignore')
    except Exception:
        return False
    return 'ATOM  ' in text or 'HETATM' in text

def load_manifest_rows(path: Path):
    if not path.exists():
        return []
    rows = []
    lines = path.read_text(errors='ignore').splitlines()
    if not lines:
        return rows
    header = lines[0].split('\t')
    for line in lines[1:]:
        vals = line.split('\t')
        rows.append(dict(zip(header, vals)))
    return rows

def candidate_paths_for(record, seq_hash_value):
    dirs = [STRUCT_DIR]
    if REUSE_EXISTING_STRUCTURES:
        dirs.insert(0, CACHE_DIR)
    if EXISTING_PDB_DIR.strip():
        dirs.insert(0, Path(EXISTING_PDB_DIR).expanduser())
    names = [
        f'{record.orf_id}__{seq_hash_value}.pdb',
        f'{record.orf_id}_{seq_hash_value}.pdb',
        f'{seq_hash_value}.pdb',
        f'{record.orf_id}.pdb',
        f'{record.orf_id}_rank_1.pdb',
    ]
    seen = set()
    candidates = []
    for directory in dirs:
        for name in names:
            path = directory / name
            key = str(path)
            if key not in seen:
                candidates.append(path)
                seen.add(key)
        for pattern in [f'{record.orf_id}*rank*.pdb', f'{record.orf_id}*.pdb', f'*{seq_hash_value}*.pdb']:
            for path in sorted(directory.glob(pattern)) if directory.exists() else []:
                key = str(path)
                if key not in seen:
                    candidates.append(path)
                    seen.add(key)
    return candidates

def register_structure(record, source_path: Path, source_label: str, seq_hash_value: str):
    if not source_path.exists() or not pdb_has_atoms(source_path):
        return False
    dst = STRUCT_DIR / f'{record.orf_id}.pdb'
    if source_path.resolve() != dst.resolve():
        shutil.copy2(source_path, dst)
    cache_path = ''
    if REUSE_EXISTING_STRUCTURES:
        cache_dst = CACHE_DIR / f'{record.orf_id}__{seq_hash_value}.pdb'
        if dst.resolve() != cache_dst.resolve():
            shutil.copy2(dst, cache_dst)
        cache_path = str(cache_dst)
    structure_paths[record.orf_id] = dst
    structure_manifest.append({
        'protein_id': record.orf_id,
        'seq_hash': seq_hash_value,
        'source': source_label,
        'path': str(dst),
        'cache_path': cache_path,
        'sequence_length': str(len(structure_sequence(record))),
    })
    return True

def find_existing_structure(record, seq_hash_value):
    manifest_rows = load_manifest_rows(STRUCT_DIR / 'structure_manifest.tsv')
    for row in manifest_rows:
        if row.get('protein_id') == record.orf_id and row.get('seq_hash') == seq_hash_value:
            for key in ['path', 'cache_path']:
                path = Path(row.get(key, ''))
                if path.exists() and pdb_has_atoms(path):
                    return path, 'manifest_cache'
    for path in candidate_paths_for(record, seq_hash_value):
        if path.exists() and pdb_has_atoms(path):
            hashed = seq_hash_value in path.name
            source = 'sequence_cache' if hashed else 'id_match_pdb'
            return path, source
    return None, ''

def patch_tensorflow_for_colabfold():
    removed = []
    patterns = [
        '/usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so',
        '/usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so',
        '/usr/local/lib/python3.*/site-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so',
        '/usr/local/lib/python3.*/site-packages/tensorflow/lite/python/*/*.so',
    ]
    for pattern in patterns:
        for path in glob.glob(pattern):
            try:
                os.remove(path)
                removed.append(path)
            except OSError:
                pass
    if removed:
        (STRUCT_DIR / 'tensorflow_colabfold_patch.log').write_text('\n'.join(removed) + '\n')

def colabfold_import_ready() -> bool:
    patch_tensorflow_for_colabfold()
    importlib.invalidate_caches()
    try:
        import colabfold.batch  # noqa: F401
        import colabfold.download  # noqa: F401
        import alphafold  # noqa: F401
        return True
    except Exception:
        return False

def ensure_colabfold_ready():
    if colabfold_import_ready():
        return
    print('Installing ColabFold AF2 backend from the ColabFold notebook...')
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts',
        'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold',
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    (STRUCT_DIR / 'colabfold_install.log').write_text((result.stdout or '') + (result.stderr or ''))
    patch_tensorflow_for_colabfold()
    importlib.invalidate_caches()
    if result.returncode != 0:
        print((result.stderr or result.stdout)[-3000:])
        raise RuntimeError('ColabFold install failed. See structures/colabfold_install.log')
    if not colabfold_import_ready():
        raise RuntimeError('ColabFold import failed after install. See structures/colabfold_install.log')

def apply_af2_preset():
    recycles = AF2_NUM_RECYCLES
    num_models = int(AF2_NUM_MODELS)
    num_seeds = int(AF2_NUM_SEEDS)
    relax_n = int(AF2_RELAX_TOP_N)
    if AF2_PRESET == 'fast':
        recycles = '3' if recycles == 'auto' else recycles
        num_models = min(num_models, 1)
        num_seeds = min(num_seeds, 1)
        relax_n = 0
    elif AF2_PRESET == 'high_accuracy':
        recycles = '12' if recycles == 'auto' else recycles
        num_models = max(num_models, 5)
        num_seeds = max(num_seeds, 1)
    else:
        recycles = '3' if recycles == 'auto' else recycles
        num_models = max(num_models, 5)
    return None if recycles == 'auto' else int(recycles), num_models, num_seeds, relax_n

def ensure_amber_ready():
    try:
        import openmm  # noqa: F401
        import pdbfixer  # noqa: F401
        return
    except Exception:
        pass
    if shutil.which('mamba') is None:
        raise RuntimeError('AF2_RELAX_TOP_N requires OpenMM/PDBFixer. Re-run Setup so mamba is available, or set AF2_RELAX_TOP_N=0.')
    print('Installing AMBER relaxation dependencies...')
    cmd = ['mamba', 'install', '-q', '-y', '-c', 'conda-forge', 'openmm=8.2.0', 'pdbfixer']
    result = subprocess.run(cmd, text=True, capture_output=True)
    (STRUCT_DIR / 'amber_install.log').write_text((result.stdout or '') + (result.stderr or ''))
    if result.returncode != 0:
        print((result.stderr or result.stdout)[-3000:])
        raise RuntimeError('AMBER dependency install failed. See structures/amber_install.log')

def predict_with_esmfold_api(record, seq_hash_value):
    seq = structure_sequence(record)
    if len(seq) > int(ESMFOLD_API_MAX_AA):
        raise RuntimeError(f'ESMFold API skipped: sequence length {len(seq)} > {ESMFOLD_API_MAX_AA} aa')
    url = 'https://api.esmatlas.com/foldSequence/v1/pdb/'
    req = urllib.request.Request(
        url,
        data=seq.encode('utf-8'),
        headers={'Content-Type': 'text/plain', 'User-Agent': 'dad-protocol-colab/1.0'},
        method='POST',
    )
    print(f'Predicting {record.orf_id} with ESMFold API...')
    try:
        with urllib.request.urlopen(req, timeout=900) as response:
            pdb_text = response.read().decode('utf-8', errors='replace')
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')[-1000:]
        (STRUCT_DIR / 'esmfold_api_error.log').write_text(detail)
        raise RuntimeError(f'ESMFold API failed with HTTP {exc.code}')
    except Exception as exc:
        raise RuntimeError(f'ESMFold API request failed: {exc}')
    if 'ATOM  ' not in pdb_text and 'HETATM' not in pdb_text:
        (STRUCT_DIR / 'esmfold_api_error.log').write_text(pdb_text[-3000:])
        raise RuntimeError('ESMFold API returned no PDB atoms')
    dst = STRUCT_DIR / f'{record.orf_id}.pdb'
    dst.write_text(pdb_text)
    register_structure(record, dst, 'esmfold_api', seq_hash_value)

def predict_with_colabfold(record, seq_hash_value):
    ensure_colabfold_ready()
    from colabfold.batch import get_queries, run, set_model_type
    from colabfold.download import download_alphafold_params
    from colabfold.utils import setup_logging

    seq = structure_sequence(record)
    job_dir = STRUCT_DIR / 'colabfold_jobs' / f'{record.orf_id}_{seq_hash_value}'
    job_dir.mkdir(parents=True, exist_ok=True)
    query_csv = job_dir / f'{record.orf_id}.csv'
    query_csv.write_text(f'id,sequence\n{record.orf_id},{seq}\n')
    setup_logging(job_dir / 'log.txt')

    queries, is_complex = get_queries(str(query_csv))
    resolved_model_type = set_model_type(is_complex, AF2_MODEL_TYPE)
    num_recycles, num_models, num_seeds, relax_n = apply_af2_preset()
    if relax_n > 0:
        ensure_amber_ready()
    use_cluster_profile = 'multimer' not in resolved_model_type
    model_order = [1, 2, 3, 4, 5][:num_models]

    print(f'Predicting {record.orf_id} with ColabFold AF2: model={resolved_model_type}, models={num_models}, recycles={num_recycles}, seeds={num_seeds}, relax={relax_n}')
    download_alphafold_params(resolved_model_type, Path('.'))
    kwargs = dict(
        queries=queries,
        result_dir=str(job_dir),
        use_templates=False,
        custom_template_path=None,
        num_relax=relax_n,
        msa_mode='mmseqs2_uniref_env',
        model_type=resolved_model_type,
        num_models=num_models,
        num_recycles=num_recycles,
        recycle_early_stop_tolerance=None,
        relax_max_iterations=200,
        num_seeds=num_seeds,
        use_dropout=False,
        model_order=model_order,
        is_complex=is_complex,
        data_dir=Path('.'),
        keep_existing_results=False,
        rank_by='auto',
        pair_mode='unpaired_paired',
        pairing_strategy='greedy',
        stop_at_score=100.0,
        prediction_callback=None,
        dpi=200,
        zip_results=False,
        save_all=False,
        max_msa=None,
        use_cluster_profile=use_cluster_profile,
        input_features_callback=None,
        save_recycles=False,
        user_agent='dad-protocol-colabfold',
        calc_extra_ptm=False,
    )
    try:
        run(**kwargs)
    except TypeError as exc:
        if 'calc_extra_ptm' in str(exc):
            kwargs.pop('calc_extra_ptm', None)
            run(**kwargs)
        else:
            raise

    pdbs = (
        sorted(job_dir.glob('*relaxed_rank_001_*.pdb')) or
        sorted(job_dir.glob('*unrelaxed_rank_001_*.pdb')) or
        sorted(job_dir.glob('*rank_001*.pdb')) or
        sorted(job_dir.glob('*.pdb'))
    )
    if not pdbs:
        raise RuntimeError(f'ColabFold produced no PDB for {record.orf_id}. See {job_dir / "log.txt"}')
    register_structure(record, pdbs[0], 'colabfold_af2', seq_hash_value)

for record in accepted:
    seq_hash_value = sequence_hash(structure_sequence(record))
    found, source = find_existing_structure(record, seq_hash_value) if REUSE_EXISTING_STRUCTURES else (None, '')
    if found:
        register_structure(record, found, source, seq_hash_value)
        print(f'{record.orf_id}: reused {found}')
    p.update(record.orf_id)

missing = [record for record in accepted if record.orf_id not in structure_paths]

for record in missing:
    seq_hash_value = sequence_hash(structure_sequence(record))
    mode = STRUCTURE_MODE
    used = False
    if mode in ('auto', 'esmfold_api'):
        try:
            predict_with_esmfold_api(record, seq_hash_value)
            used = True
        except Exception as exc:
            print(f'{record.orf_id}: ESMFold unavailable ({exc}).')
            if mode == 'esmfold_api' and not ALLOW_COLABFOLD_FALLBACK:
                raise
    if not used and mode != 'existing_or_upload':
        if mode == 'colabfold' or (mode in ('auto', 'esmfold_api') and ALLOW_COLABFOLD_FALLBACK):
            predict_with_colabfold(record, seq_hash_value)
            used = True
    if not used and mode == 'existing_or_upload':
        print(f'{record.orf_id}: prediction skipped; waiting for uploaded PDB.')
    p.update(f'predicted {record.orf_id}' if used else f'missing {record.orf_id}')

p.update('structure search complete')

missing_ids = [record.orf_id for record in accepted if record.orf_id not in structure_paths]
if missing_ids:
    raise RuntimeError(
        'No PDB structure for: ' + ', '.join(missing_ids) + '. '
        'Use STRUCTURE_MODE="auto" or upload matching PDB files.'
    )

manifest_path = STRUCT_DIR / 'structure_manifest.tsv'
columns = ['protein_id', 'seq_hash', 'source', 'path', 'cache_path', 'sequence_length']
manifest_path.write_text('\t'.join(columns) + '\n' + '\n'.join('\t'.join(row.get(col, '') for col in columns) for row in structure_manifest) + '\n')

for prot_id, path in structure_paths.items():
    print(f'{prot_id}: {path}')
print(f'Cache: {CACHE_DIR}')
print(f'Manifest: {manifest_path}')
p.done(f'{len(structure_paths)} structure(s) ready')


In [ ]:
#@title 4. Pocket and Box Setup {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** docking box center and size per protein.

BOX_CENTER_MODE = "protein_center"  #@param ["protein_center"]
BOX_SIZE_A = 30.0  #@param {type:"number"}

import numpy as np
from pathlib import Path

p = dad_progress('Pocket/Box', max(1, len(structure_paths)))
BOX_SIZE = float(BOX_SIZE_A)
pocket_centers = {}

def read_pdb_coords(path: Path) -> np.ndarray:
    coords = []
    for line in Path(path).read_text(errors='ignore').splitlines():
        if line.startswith(('ATOM', 'HETATM')) and len(line) >= 54:
            try:
                coords.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            except ValueError:
                pass
    if not coords:
        raise ValueError(f'No atom coordinates parsed from {path}')
    return np.array(coords, dtype=float)

for prot_id, pdb_path in structure_paths.items():
    coords = read_pdb_coords(pdb_path)
    center = coords.mean(axis=0)
    span = coords.max(axis=0) - coords.min(axis=0)
    auto_size = max(BOX_SIZE, min(60.0, float(span.max()) + 6.0))
    pocket_centers[prot_id] = {'x': float(center[0]), 'y': float(center[1]), 'z': float(center[2]), 'size': auto_size}
    print(f'{prot_id}: center=({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}), box={auto_size:.1f} A')
    p.update(prot_id)

p.done('complete')


In [ ]:
#@title 5. Ligand Preparation {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** 3D SDF files and docking case list.

from rdkit import Chem
from rdkit.Chem import AllChem, SDWriter

p = dad_progress('Ligand Preparation', max(1, len(LIGANDS) + 1))
LIGAND_DIR = WORK_DIR / 'ligands_3d'
LIGAND_DIR.mkdir(parents=True, exist_ok=True)
ligand_sdf_paths = {}

for name, smiles in LIGANDS.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Could not parse SMILES for {name}: {smiles}')
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 0
    result = AllChem.EmbedMolecule(mol, params)
    if result != 0:
        result = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    if result != 0:
        raise RuntimeError(f'3D embedding failed for ligand {name}')
    AllChem.MMFFOptimizeMolecule(mol)
    sdf_path = LIGAND_DIR / f'{name}.sdf'
    writer = SDWriter(str(sdf_path))
    mol.SetProp('_Name', name)
    writer.write(mol)
    writer.close()
    ligand_sdf_paths[name] = sdf_path
    print(f'{name}: {sdf_path}')
    p.update(name)

cases = [{'protein': prot_id, 'ligand': lig_id} for prot_id in structure_paths for lig_id in ligand_sdf_paths]
p.update('case list')
print(f'Total docking cases: {len(cases)}')
for case in cases:
    print(f'{case["protein"]} x {case["ligand"]}')
p.done('complete')


In [ ]:
#@title 6. GNINA Docking {display-mode: "form"}
#@markdown **Time:** depends on receptor size, box size, and GPU.
#@markdown **Output:** docked SDF, GNINA log, score record per case.

GNINA_EXHAUSTIVENESS = 32  #@param {type:"integer"}
GNINA_NUM_MODES = 9  #@param {type:"integer"}
GNINA_SEED = 0  #@param {type:"integer"}

import subprocess
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

p = dad_progress('GNINA Docking', max(1, len(cases)))
GNINA_BIN = Path('/usr/local/bin/gnina')

@dataclass
class DockingResult:
    case_id: str
    protein_id: str
    ligand_id: str
    vina_score: Optional[float]
    cnn_pose_score: Optional[float]
    cnn_affinity: Optional[float]
    pose_count: int
    output_sdf: str
    log_file: str
    runtime_s: float
    status: str

def parse_first_pose_props(sdf_path: Path):
    props = {'vina_score': None, 'cnn_pose_score': None, 'cnn_affinity': None, 'pose_count': 0}
    if not sdf_path.exists():
        return props
    text = sdf_path.read_text(errors='ignore')
    props['pose_count'] = text.count('$$$$')
    aliases = {'minimizedAffinity': 'vina_score', 'CNNscore': 'cnn_pose_score', 'CNNaffinity': 'cnn_affinity'}
    lines = text.splitlines()
    for idx, line in enumerate(lines[:-1]):
        tag = line.strip().replace('>  <', '').replace('>', '').replace('<', '')
        if tag in aliases:
            try:
                props[aliases[tag]] = float(lines[idx + 1].strip())
            except ValueError:
                pass
    return props

def run_gnina(case):
    prot_id = case['protein']
    lig_id = case['ligand']
    case_id = f'{prot_id}_x_{lig_id}'
    out_dir = WORK_DIR / 'docking' / case_id
    out_dir.mkdir(parents=True, exist_ok=True)
    out_sdf = out_dir / 'docked.sdf'
    log_file = out_dir / 'gnina.log'
    center = pocket_centers[prot_id]
    box = center.get('size', BOX_SIZE)
    cmd = [
        str(GNINA_BIN), '-r', str(structure_paths[prot_id]), '-l', str(ligand_sdf_paths[lig_id]),
        '--center_x', str(center['x']), '--center_y', str(center['y']), '--center_z', str(center['z']),
        '--size_x', str(box), '--size_y', str(box), '--size_z', str(box),
        '--exhaustiveness', str(GNINA_EXHAUSTIVENESS), '--num_modes', str(GNINA_NUM_MODES),
        '--seed', str(GNINA_SEED), '--out', str(out_sdf),
    ]
    start = time.time()
    proc = subprocess.run(cmd, text=True, capture_output=True)
    runtime = time.time() - start
    log_file.write_text(proc.stdout + proc.stderr)
    props = parse_first_pose_props(out_sdf)
    status = 'DONE' if proc.returncode == 0 and out_sdf.exists() and props['pose_count'] else 'FAIL'
    if status == 'FAIL':
        print((proc.stderr or proc.stdout)[-1500:])
    return DockingResult(case_id, prot_id, lig_id, props['vina_score'], props['cnn_pose_score'],
                         props['cnn_affinity'], int(props['pose_count'] or 0), str(out_sdf),
                         str(log_file), runtime, status)

if EXEC_MODE == 'prepare_only':
    docking_results = []
    print('prepare_only mode: docking skipped after input/structure/ligand preparation.')
elif not GNINA_BIN.exists():
    raise RuntimeError('GNINA not found. Run Setup with INSTALL_GNINA=True or set exec_mode="prepare_only".')
else:
    docking_results = []
    for case in cases:
        result = run_gnina(case)
        docking_results.append(result)
        print(f'{result.case_id}: {result.status}, poses={result.pose_count}, cnn_pose={result.cnn_pose_score}')
        p.update(result.case_id)

p.done(f'{len(docking_results)} docking result(s)')


In [ ]:
#@title 7. Quantitative Analysis {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** ranked user docking results.

import pandas as pd

p = dad_progress('Analysis', 4)
rows = [vars(result) for result in docking_results]
df_results = pd.DataFrame(rows)
p.update('dataframe')

if df_results.empty:
    df_ranked = pd.DataFrame()
    print('No docking results. If you used prepare_only, run live docking later with GNINA.')
else:
    for col in ['vina_score', 'cnn_pose_score', 'cnn_affinity', 'runtime_s']:
        df_results[col] = pd.to_numeric(df_results[col], errors='coerce')
    df_ranked = df_results.sort_values(
        by=['cnn_pose_score', 'cnn_affinity', 'vina_score'],
        ascending=[False, False, True],
        na_position='last',
    ).reset_index(drop=True)
    df_ranked.insert(0, 'rank', range(1, len(df_ranked) + 1))
    p.update('ranking')
    display_cols = ['rank', 'case_id', 'status', 'cnn_pose_score', 'cnn_affinity', 'vina_score', 'pose_count', 'runtime_s']
    print(df_ranked[display_cols].to_string(index=False))
    p.update('summary')
    out_path = WORK_DIR / 'results' / 'docking_results.tsv'
    df_ranked.to_csv(out_path, sep='\t', index=False)
    print(f'Results saved: {out_path}')
    p.update('saved')

p.done('complete')


In [ ]:
#@title 8. Structure Visualization {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** py3Dmol view for top docked cases.

MAX_VIEW_CASES = 3  #@param {type:"integer"}

p = dad_progress('Visualization', max(1, min(MAX_VIEW_CASES, len(docking_results) if 'docking_results' in globals() else 0)))

try:
    import py3Dmol
    PY3DMOL_AVAILABLE = True
except ImportError:
    PY3DMOL_AVAILABLE = False
    print('py3Dmol not available.')

def visualize_docking(result):
    if not PY3DMOL_AVAILABLE:
        return
    pdb_path = Path(structure_paths[result.protein_id])
    sdf_path = Path(result.output_sdf)
    if not pdb_path.exists() or not sdf_path.exists():
        print(f'{result.case_id}: missing PDB or SDF')
        return
    view = py3Dmol.view(width=800, height=500)
    view.addModel(pdb_path.read_text(errors='ignore'), 'pdb')
    view.setStyle({'cartoon': {'color': 'lightblue'}})
    view.addModel(sdf_path.read_text(errors='ignore'), 'sdf')
    view.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo({'model': -1})
    view.show()
    print(f'{result.case_id}: {result.status}')

if docking_results:
    for result in docking_results[:MAX_VIEW_CASES]:
        visualize_docking(result)
        p.update(result.case_id)
else:
    print('No docked SDF files to visualize.')

p.done('complete')


In [ ]:
#@title 9. Result Tables {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** ranked, protein-level, and ligand-level TSV files.

import pandas as pd

p = dad_progress('Result Tables', 4)
TABLES_DIR = WORK_DIR / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if 'df_ranked' not in globals() or df_ranked.empty:
    print('No ranked docking table available.')
else:
    ranked_path = TABLES_DIR / 'ranked_results.tsv'
    df_ranked.to_csv(ranked_path, sep='\t', index=False)
    p.update('ranked')
    protein_summary = df_ranked.groupby('protein_id').agg(
        n_cases=('case_id', 'count'), best_cnn_pose=('cnn_pose_score', 'max'),
        best_cnn_affinity=('cnn_affinity', 'max'), best_vina=('vina_score', 'min'),
    ).reset_index()
    protein_path = TABLES_DIR / 'protein_summary.tsv'
    protein_summary.to_csv(protein_path, sep='\t', index=False)
    p.update('protein summary')
    ligand_summary = df_ranked.groupby('ligand_id').agg(
        n_cases=('case_id', 'count'), best_cnn_pose=('cnn_pose_score', 'max'),
        best_cnn_affinity=('cnn_affinity', 'max'), best_vina=('vina_score', 'min'),
    ).reset_index()
    ligand_path = TABLES_DIR / 'ligand_summary.tsv'
    ligand_summary.to_csv(ligand_path, sep='\t', index=False)
    p.update('ligand summary')
    print(f'Saved: {ranked_path}')
    print(f'Saved: {protein_path}')
    print(f'Saved: {ligand_path}')
    p.update('saved')

p.done('complete')


In [ ]:
#@title 10. Result Figures {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** score plots for user docking results.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

p = dad_progress('Result Figures', 3)
FIGS_DIR = WORK_DIR / 'results' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

if 'df_ranked' not in globals() or df_ranked.empty:
    print('No ranked docking table available for plotting.')
else:
    plot_df = df_ranked.copy()
    labels = plot_df['case_id'].astype(str).tolist()
    x = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(max(6, len(plot_df) * 0.8), 4))
    ax.bar(x, plot_df['cnn_pose_score'].fillna(0), color='#2E86AB')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('CNN pose score')
    ax.set_title('Docking score ranking')
    fig.tight_layout()
    fig.savefig(FIGS_DIR / 'cnn_pose_ranking.png', dpi=300)
    fig.savefig(FIGS_DIR / 'cnn_pose_ranking.pdf')
    plt.show()
    p.update('bar plot')
    if plot_df['protein_id'].nunique() > 1 or plot_df['ligand_id'].nunique() > 1:
        heat = plot_df.pivot_table(index='protein_id', columns='ligand_id', values='cnn_pose_score', aggfunc='max')
        fig, ax = plt.subplots(figsize=(max(5, heat.shape[1] * 1.2), max(4, heat.shape[0] * 0.8)))
        im = ax.imshow(heat.fillna(0).values, cmap='viridis', aspect='auto')
        ax.set_xticks(np.arange(heat.shape[1]))
        ax.set_xticklabels(heat.columns, rotation=45, ha='right')
        ax.set_yticks(np.arange(heat.shape[0]))
        ax.set_yticklabels(heat.index)
        ax.set_title('Protein-ligand CNN pose score matrix')
        fig.colorbar(im, ax=ax, label='CNN pose score')
        fig.tight_layout()
        fig.savefig(FIGS_DIR / 'score_matrix.png', dpi=300)
        fig.savefig(FIGS_DIR / 'score_matrix.pdf')
        plt.show()
    p.update('matrix plot')
    if IN_COLAB:
        from google.colab import files
        import zipfile
        zip_path = WORK_DIR / 'results' / 'figures.zip'
        with zipfile.ZipFile(str(zip_path), 'w') as zf:
            for file in FIGS_DIR.glob('*'):
                zf.write(str(file), file.name)
        files.download(str(zip_path))
        print(f'Download started: {zip_path}')
    p.update('saved')

p.done('complete')


In [ ]:
#@title 11. Download Results {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** result archive.

import zipfile

p = dad_progress('Download Results', 3)
RESULTS_DIR = WORK_DIR / 'results'
archive_path = WORK_DIR / f'{WORK_DIR.name}_results.zip'
with zipfile.ZipFile(str(archive_path), 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file in RESULTS_DIR.rglob('*'):
        if file.is_file():
            zf.write(str(file), file.relative_to(WORK_DIR))
p.update('archive')
print(f'Archive saved: {archive_path}')
p.update('ready')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))
    print('Download started.')
p.update('download')
p.done('complete')


In [ ]:
#@title 12. Reproducibility Footprint {display-mode: "form"}
#@markdown **Time:** <1 min.
#@markdown **Output:** reproducibility_footprint.json.

import importlib.metadata
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone

p = dad_progress('Reproducibility', 4)
footprint = {
    'recorded_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'exec_mode': EXEC_MODE,
    'in_colab': IN_COLAB,
    'n_proteins': len(PROTEINS),
    'n_ligands': len(LIGANDS),
    'n_cases': len(cases) if 'cases' in globals() else 0,
    'n_docking_results': len(docking_results) if 'docking_results' in globals() else 0,
    'work_dir': str(WORK_DIR),
    'structure_mode': globals().get('STRUCTURE_MODE', 'unknown'),
    'structure_cache_dir': str(globals().get('CACHE_DIR', '')),
    'structure_paths': {k: str(v) for k, v in globals().get('structure_paths', {}).items()},
}
p.update('metadata')
for pkg in ['rdkit', 'pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn', 'biopython', 'py3Dmol']:
    try:
        footprint[f'pkg_{pkg}'] = importlib.metadata.version(pkg)
    except Exception:
        footprint[f'pkg_{pkg}'] = 'unknown'
p.update('packages')
try:
    result = subprocess.run(['/usr/local/bin/gnina', '--version'], capture_output=True, text=True, timeout=5)
    footprint['gnina_version'] = (result.stdout + result.stderr).strip()[:200]
except Exception:
    footprint['gnina_version'] = 'not available'
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True, timeout=5)
    footprint['gpu'] = result.stdout.strip()
except Exception:
    footprint['gpu'] = 'not available'
p.update('runtime')
fp_path = WORK_DIR / 'results' / 'reproducibility_footprint.json'
fp_path.write_text(json.dumps(footprint, indent=2))
print(f'Footprint saved: {fp_path}')
print(json.dumps({k: footprint[k] for k in ['exec_mode', 'n_proteins', 'n_ligands', 'n_cases', 'n_docking_results', 'gnina_version', 'gpu']}, indent=2))
p.update('saved')
p.done('complete')
